### Get Junior author data from open alex

In [1]:
import pandas as pd
import requests
import json
import time

MAILTO = 'shaheryar.4822@student.uu.se'  # ← replace

# ── Load junior authors ────────────────────────────────────────────────────────
df = pd.read_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\junior_authors_all_conferences.csv')

# Deduplicate: one profile per unique author
# (same author may appear across multiple awards)
unique_authors = df.drop_duplicates(subset='author_id').copy()
print(f"Unique junior authors to profile: {len(unique_authors)}")
print(f"(from {len(df)} total junior-award records)")

# ── Helper: fetch full author profile ─────────────────────────────────────────
def get_author_profile(author_id):
    """Fetch author-level metrics from OpenAlex."""
    url = f"https://api.openalex.org/authors/{author_id.split('/')[-1]}"
    try:
        r = requests.get(url, params={'mailto': MAILTO}, timeout=10)
        if r.status_code == 200:
            return r.json()
    except:
        pass
    return None

def get_counts_by_year(author_id):
    """Fetch yearly works+citations breakdown."""
    url = f"https://api.openalex.org/authors/{author_id.split('/')[-1]}"
    try:
        r = requests.get(url, params={'mailto': MAILTO, 'select': 'counts_by_year'}, timeout=10)
        if r.status_code == 200:
            return r.json().get('counts_by_year', [])
    except:
        pass
    return []

def get_first_pub_year(author_id):
    """Get earliest publication year via works endpoint."""
    url = 'https://api.openalex.org/works'
    params = {
        'filter': f'author.id:{author_id}',
        'sort': 'publication_year:asc',
        'per-page': 1,
        'mailto': MAILTO
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200:
            results = r.json().get('results', [])
            if results:
                return results[0].get('publication_year')
    except:
        pass
    return None

# ── Main profiling loop ────────────────────────────────────────────────────────
profiles = []

for i, row in unique_authors.iterrows():
    author_id = row['author_id']

    profile = get_author_profile(author_id)
    time.sleep(0.12)

    if not profile:
        print(f"[{len(profiles)+1}/{len(unique_authors)}] ✗ No profile: {row['author_name']}")
        continue

    # Extract core metrics
    summary_stats = profile.get('summary_stats', {})
    counts_by_year = profile.get('counts_by_year', [])

    # Get first pub year from profile, fallback to works endpoint
    first_pub_year = profile.get('works_count')  # placeholder
    affiliations = profile.get('affiliations', [])
    first_pub_year = None
    for aff in affiliations:
        years = aff.get('years', [])
        if years:
            candidate = min(years)
            if first_pub_year is None or candidate < first_pub_year:
                first_pub_year = candidate

    # Fallback: use works endpoint for first pub year
    if not first_pub_year:
        first_pub_year = get_first_pub_year(author_id)
        time.sleep(0.08)

    profiles.append({
        'author_id':            author_id,
        'author_name':          row['author_name'],
        'award_year':           row['award_year'],
        'conference':           row['conference'],
        'career_age_at_award':  row['career_age_at_award'],
        'author_position':      row['author_position'],
        'match_route':          row.get('match_route', ''),
        # Metrics
        'works_count':          profile.get('works_count', 0),
        'cited_by_count':       profile.get('cited_by_count', 0),
        'h_index':              summary_stats.get('h_index', 0),
        'i10_index':            summary_stats.get('i10_index', 0),
        '2yr_mean_citedness':   summary_stats.get('2yr_mean_citedness', 0),
        'first_pub_year':       first_pub_year,
        'counts_by_year':       json.dumps(counts_by_year),
    })

    if len(profiles) % 50 == 0:
        pd.DataFrame(profiles).to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\profiles\\junior_profiles_progress.csv', index=False)
        print(f"[{len(profiles)}/{len(unique_authors)}] Saved checkpoint...")

# ── Final save ─────────────────────────────────────────────────────────────────
profiles_df = pd.DataFrame(profiles)
profiles_df.to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\profiles\\junior_profiles_all.csv', index=False)

print("=" * 60)
print(f"Profiles fetched:   {len(profiles_df)}")
print(f"Missing:            {len(unique_authors) - len(profiles_df)}")
print(f"\nh_index stats:\n{profiles_df['h_index'].describe()}")
print(f"\nConference breakdown:\n{profiles_df['conference'].value_counts().head(15)}")


Unique junior authors to profile: 603
(from 603 total junior-award records)
[50/603] Saved checkpoint...
[100/603] Saved checkpoint...
[150/603] Saved checkpoint...
[200/603] Saved checkpoint...
[250/603] Saved checkpoint...
[300/603] Saved checkpoint...
[350/603] Saved checkpoint...
[400/603] Saved checkpoint...
[450/603] Saved checkpoint...
[500/603] Saved checkpoint...
[550/603] Saved checkpoint...
[600/603] Saved checkpoint...
Profiles fetched:   603
Missing:            0

h_index stats:
count    603.000000
mean      14.381426
std       12.865279
min        0.000000
25%        5.000000
50%       11.000000
75%       19.000000
max       92.000000
Name: h_index, dtype: float64

Conference breakdown:
conference
CHI        119
ICSE        81
FSE         47
UIST        27
PLDI        26
OSDI        22
SOSP        22
ACL         19
VLDB        18
AAAI        16
WWW         16
INFOCOM     16
SIGCOMM     12
KDD         12
SIGMOD      12
Name: count, dtype: int64
